# Multi-Relational R-GCN for Eviction Rate Prediction

Implements and evaluates R-GCN with two hand-designed relation types:
- **Relation 0** — Cosine similarity kNN (K=1, threshold=0.3)
- **Relation 1** — Queen contiguity from TIGER/Line shapefiles

**Temporal split:**
- **Train:** ACS 2022 features → predict eviction rate 2024
- **Test:**  ACS 2023 features → predict eviction rate 2025 (inductive)

**Experiments:**
| ID | Model | Description |
|----|-------|-------------|
| A  | GCN-Similarity  | GCN on similarity edges only |
| B  | GCN-Contiguity  | GCN on contiguity edges only |
| D  | RGCN-SimCont    | R-GCN: sim + cont (full weight matrices) |
| D2 | RGCN-SimCont-Reg | R-GCN: sim + cont (1 shared basis matrix) |
| H  | GCN-Merged      | GCN on union of sim+cont (single relation) |

**Environment:** Tested on Google Colab A100 80 GB SXM4.


## Configuration

In [ ]:
import os

# ── Data paths — edit these to match your local setup ─────────────────────
ACS_DIR   = 'data/ACS_clean'          # ACS aligned CSV files (2022, 2023)
EVT_DIR   = 'data/Eviction'           # Eviction filing CSV files (2022–2025)
SHAPE_DIR = 'data/shapefiles'         # TIGER/Line tract shapefiles
OUT_DIR   = 'outputs'                 # where results are saved

os.makedirs(OUT_DIR, exist_ok=True)

# ── Graph hyperparameters ─────────────────────────────────────────────────
SIM_K          = 1      # KNN neighbours for similarity relation
SIM_THRESHOLD  = 0.3    # cosine distance threshold for similarity edges

# ── Model hyperparameters ─────────────────────────────────────────────────
HIDDEN_DIM   = 64
EPOCHS       = 200
LR           = 1e-3
WEIGHT_DECAY = 5e-4
PATIENCE     = None       # early stopping patience (set None to disable)
VAL_FRAC     = 0.15     # fraction of train set used for validation
SEED         = 42

# ── Feature flags ─────────────────────────────────────────────────────────
USE_LAGGED = False   # set True to append lagged eviction rate as a feature


## Imports

In [ ]:
# Uncomment if running on Google Colab
# from google.colab import drive
# drive.mount('/content/drive')

import os, warnings, math
from copy import deepcopy

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, RGCNConv
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import mean_squared_error, r2_score
from scipy import stats

warnings.filterwarnings('ignore')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')
if device.type == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')


## Data Loading

In [ ]:
def to_geoid_str(series):
    return series.astype(str).str.strip().str.zfill(11)

def compute_eviction_rate(evt_df, acs_df, filings_col, pop_col, label):
    pop = acs_df[[pop_col]].rename(columns={pop_col: 'population'})
    df  = evt_df[[filings_col]].copy()
    df['population'] = pop.reindex(df.index)['population']
    n_before = len(df)
    df = df[df['population'].notna() & (df['population'] > 0)]
    df[filings_col]     = df[filings_col].fillna(0)
    df['eviction_rate'] = (df[filings_col] / df['population']) * 100
    print(f'  {label}: {n_before:,} -> {len(df):,} tracts  '
          f'mean={df["eviction_rate"].mean():.4f}  max={df["eviction_rate"].max():.4f}')
    return df['eviction_rate']


FILINGS_COL    = 'total_filings'
POPULATION_COL = 'F001'

print('Loading ACS data ...')
acs_2022 = pd.read_csv(f'{ACS_DIR}/acs_aligned_2022_5_tract.csv')
acs_2023 = pd.read_csv(f'{ACS_DIR}/acs_aligned_2023_5_tract.csv')
for df in [acs_2022, acs_2023]:
    df['FIPS'] = to_geoid_str(df['FIPS'])
    df.set_index('FIPS', inplace=True)

print('\nLoading eviction data ...')
evt = {}
for yr in [2022, 2023, 2024, 2025]:
    evt[yr] = pd.read_csv(f'{EVT_DIR}/eviction_filings_{yr}.csv')
    evt[yr]['GEOID'] = to_geoid_str(evt[yr]['GEOID'])
    evt[yr].set_index('GEOID', inplace=True)

evt_rate_2022 = compute_eviction_rate(evt[2022], acs_2022, FILINGS_COL, POPULATION_COL, '2022 rate')
evt_rate_2023 = compute_eviction_rate(evt[2023], acs_2023, FILINGS_COL, POPULATION_COL, '2023 rate')
evt_rate_2024 = compute_eviction_rate(evt[2024], acs_2022, FILINGS_COL, POPULATION_COL, '2024 rate (train label)')
evt_rate_2025 = compute_eviction_rate(evt[2025], acs_2023, FILINGS_COL, POPULATION_COL, '2025 rate (test label)')


## Train / Test Split and Preprocessing

In [ ]:
# Train: ACS 2022 -> predict eviction rate 2024
train_common = (acs_2022.index.intersection(evt_rate_2022.index)
                              .intersection(evt_rate_2024.index))
train_X = acs_2022.loc[train_common].copy()
if USE_LAGGED:
    train_X['eviction_rate_prev'] = evt_rate_2022.reindex(train_common)
train_y = evt_rate_2024.reindex(train_common)
valid   = train_y.notna()
if USE_LAGGED: valid &= train_X['eviction_rate_prev'].notna()
train_X, train_y = train_X[valid], train_y[valid]

# Test: ACS 2023 -> predict eviction rate 2025
test_common = (acs_2023.index.intersection(evt_rate_2023.index)
                             .intersection(evt_rate_2025.index))
test_X = acs_2023.loc[test_common].copy()
if USE_LAGGED:
    test_X['eviction_rate_prev'] = evt_rate_2023.reindex(test_common)
test_y = evt_rate_2025.reindex(test_common)
valid  = test_y.notna()
if USE_LAGGED: valid &= test_X['eviction_rate_prev'].notna()
test_X, test_y = test_X[valid], test_y[valid]

# Align feature columns
if USE_LAGGED:
    acs_shared   = (train_X.columns.drop('eviction_rate_prev')
                    .intersection(test_X.columns.drop('eviction_rate_prev')).tolist())
    shared_feats = acs_shared + ['eviction_rate_prev']
else:
    shared_feats = train_X.columns.intersection(test_X.columns).tolist()
train_X, test_X = train_X[shared_feats], test_X[shared_feats]

n_train, n_test = len(train_X), len(test_X)
train_geoid_to_pos = {g: i for i, g in enumerate(train_X.index)}
test_geoid_to_pos  = {g: i for i, g in enumerate(test_X.index)}
print(f'Train : {n_train:,} tracts | Test : {n_test:,} tracts | Features : {len(shared_feats)}')

# Preprocess — fit on train only
preprocessor = Pipeline([('imputer', SimpleImputer(strategy='mean')),
                          ('scaler',  StandardScaler())])
X_train_proc = preprocessor.fit_transform(train_X)
X_test_proc  = preprocessor.transform(test_X)

x_train_tensor = torch.tensor(X_train_proc, dtype=torch.float)
y_train_tensor = torch.tensor(train_y.values, dtype=torch.float).view(-1, 1)
x_test_tensor  = torch.tensor(X_test_proc,  dtype=torch.float)
y_test_tensor  = torch.tensor(test_y.values, dtype=torch.float).view(-1, 1)

# Validation split (15% of train, used for early stopping)
rng       = np.random.default_rng(SEED)
n_val     = int(n_train * VAL_FRAC)
val_idx   = rng.choice(n_train, size=n_val, replace=False)
train_idx = np.setdiff1d(np.arange(n_train), val_idx)
train_mask = torch.zeros(n_train, dtype=torch.bool); train_mask[train_idx] = True
val_mask   = torch.zeros(n_train, dtype=torch.bool); val_mask[val_idx]     = True
print(f'  train nodes={train_mask.sum():,}  val nodes={val_mask.sum():,}')


## Graph Construction

### Relation 0 — Cosine Similarity (K=1, threshold=0.3)

In [ ]:
def build_similarity_edges(X, K=SIM_K, threshold=SIM_THRESHOLD):
    """Directed cosine-kNN edges (i->j only) with distance threshold."""
    print(f'  Building similarity edges  K={K}  threshold={threshold} ...', end=' ')
    nbrs = NearestNeighbors(n_neighbors=K+1, metric='cosine', algorithm='brute')
    nbrs.fit(X)
    dist, idx = nbrs.kneighbors(X)
    src, tgt = [], []
    for i in range(len(X)):
        for j in range(1, K+1):
            if dist[i, j] < threshold:
                src.append(i); tgt.append(idx[i, j])
    ei = torch.unique(torch.tensor([src, tgt], dtype=torch.long), dim=1)
    print(f'{ei.shape[1]:,} edges')
    return ei

print('\nRelation 0 — Similarity:')
ei_sim_train = build_similarity_edges(X_train_proc)
ei_sim_test  = build_similarity_edges(X_test_proc)


### Relation 1 — Queen Contiguity (TIGER/Line shapefiles)

In [ ]:
def build_global_contiguity_edges(node_order, geoid_to_pos, X_scaled,
                                   shape_dir, fallback_K=1, fallback_t=0.3):
    """
    Queen contiguity edges, cross-state borders included (directed: i->j, i<j).
    Isolated tracts receive a fallback similarity edge.
    Returns (ei_contiguity, ei_fallback, stats_dict).
    """
    import geopandas as gpd
    from libpysal.weights import Queen as QueenW

    nbrs_fb = NearestNeighbors(n_neighbors=fallback_K+1, metric='cosine', algorithm='brute')
    nbrs_fb.fit(X_scaled)
    dist_fb, idx_fb = nbrs_fb.kneighbors(X_scaled)

    states = sorted(set(g[:2] for g in node_order))
    gdfs, missing = [], []
    print('  Loading shapefiles ...')
    for st in states:
        shp = os.path.join(shape_dir, f'tl_2022_{st}_tract.shp')
        if not os.path.exists(shp):
            shp = os.path.join(shape_dir, f'tl_2023_{st}_tract.shp')
        if not os.path.exists(shp):
            missing.append(st); continue
        gdf = gpd.read_file(shp)
        gdf['GEOID'] = (gdf['STATEFP'].str.zfill(2) +
                        gdf['COUNTYFP'].str.zfill(3) +
                        gdf['TRACTCE'].str.zfill(6))
        gdfs.append(gdf[['GEOID', 'geometry']])
    print(f'  Loaded {len(gdfs)} states ({len(missing)} missing: {missing})')

    gdf_global = (pd.concat(gdfs, ignore_index=True)
                  [lambda d: d['GEOID'].isin(set(node_order))]
                  .reset_index(drop=True))
    print(f'  Global GDF : {len(gdf_global):,} tracts')

    print('  Computing global Queen adjacency ... (may take ~1 min)')
    w = QueenW.from_dataframe(gdf_global, silence_warnings=True)

    cont_src, cont_tgt = [], []
    fall_src, fall_tgt = [], []
    isolated = []
    for row_pos in range(len(gdf_global)):
        geoid = gdf_global['GEOID'].iloc[row_pos]
        if geoid not in geoid_to_pos: continue
        i = geoid_to_pos[geoid]
        valid_js = [geoid_to_pos[gdf_global['GEOID'].iloc[nb]]
                    for nb in w.neighbors.get(row_pos, [])
                    if gdf_global['GEOID'].iloc[nb] in geoid_to_pos]
        if not valid_js:
            isolated.append(i)
        else:
            for j in valid_js:
                cont_src.append(i); cont_tgt.append(j)
    for i in isolated:
        if dist_fb[i, 1] < fallback_t:
            fall_src.append(i); fall_tgt.append(idx_fb[i, 1])

    ei_cont = torch.tensor([cont_src, cont_tgt], dtype=torch.long)
    ei_fall = torch.tensor([fall_src, fall_tgt], dtype=torch.long)
    print(f'  Contiguity: {ei_cont.shape[1]:,}  Fallback: {ei_fall.shape[1]:,}  Isolated: {len(isolated):,}')
    return ei_cont, ei_fall, {}


print('\nRelation 1 — Contiguity (train):')
ei_cont_train, ei_fallback_train, _ = build_global_contiguity_edges(
    train_X.index, train_geoid_to_pos, X_train_proc, SHAPE_DIR)

print('\nRelation 1 — Contiguity (test):')
ei_cont_test, ei_fallback_test, _ = build_global_contiguity_edges(
    test_X.index, test_geoid_to_pos, X_test_proc, SHAPE_DIR)

# Merge fallback into similarity relation
ei_sim_train_full = torch.unique(torch.cat([ei_sim_train, ei_fallback_train], dim=1), dim=1)
ei_sim_test_full  = torch.unique(torch.cat([ei_sim_test,  ei_fallback_test],  dim=1), dim=1)


## Model Definitions

In [ ]:
class GCNModel(nn.Module):
    """Two-layer GCN for single-relation regression."""
    def __init__(self, in_dim, hidden_dim=HIDDEN_DIM, dropout=0.5):
        super().__init__()
        self.conv1   = GCNConv(in_dim, hidden_dim)
        self.conv2   = GCNConv(hidden_dim, 1)
        self.dropout = dropout

    def forward(self, x, edge_index):
        h = F.relu(self.conv1(x, edge_index))
        h = F.dropout(h, p=self.dropout, training=self.training)
        return self.conv2(h, edge_index)


class RGCNModel(nn.Module):
    """Two-layer R-GCN for multi-relational regression."""
    def __init__(self, in_dim, hidden_dim=HIDDEN_DIM,
                 num_relations=2, num_bases=None, dropout=0.5):
        super().__init__()
        self.conv1   = RGCNConv(in_dim, hidden_dim,
                                num_relations=num_relations,
                                num_bases=num_bases)
        self.conv2   = RGCNConv(hidden_dim, 1,
                                num_relations=num_relations,
                                num_bases=num_bases)
        self.dropout = dropout

    def forward(self, x, edge_index, edge_type):
        h = F.relu(self.conv1(x, edge_index, edge_type))
        h = F.dropout(h, p=self.dropout, training=self.training)
        return self.conv2(h, edge_index, edge_type)


## Training and Evaluation Helpers

In [ ]:
criterion = nn.MSELoss()
BINS = [(0,1,'0-1'),(1,5,'1-5'),(5,10,'5-10'),
        (10,20,'10-20'),(20,25,'20-25'),(25,float('inf'),'25+')]


def set_seed(seed=SEED):
    import random
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True


def compute_metrics(true, pred, prefix=''):
    rmse = math.sqrt(mean_squared_error(true, pred))
    mae  = np.abs(true - pred).mean()
    r2   = r2_score(true, pred)
    return {f'{prefix}rmse': rmse, f'{prefix}mae': mae, f'{prefix}r2': r2}


def print_binwise(true, pred, bins=BINS):
    print(f'  {"Bin":>8}  {"N":>6}  {"RMSE":>8}')
    for lo, hi, label in bins:
        mask = (true >= lo) & (true < hi)
        if mask.sum() == 0: continue
        rmse_b = math.sqrt(((pred[mask] - true[mask])**2).mean())
        print(f'  {label:>8}  {mask.sum():>6,}  {rmse_b:>8.4f}')


def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def train_eval_gcn(label, model, ei_train, ei_val, ei_test,
                   x_tr, y_tr, x_va, y_va, x_te, y_te,
                   tr_mask, va_mask, verbose=True):
    """Train single-relation GCN with early stopping on validation MSE."""
    model = model.to(device)
    x_tr, y_tr = x_tr.to(device), y_tr.to(device)
    x_va, y_va = x_va.to(device), y_va.to(device)
    x_te, y_te = x_te.to(device), y_te.to(device)
    ei_tr = ei_train.to(device)
    ei_va = ei_val.to(device)
    ei_te = ei_test.to(device)
    tr_m, va_m = tr_mask.to(device), va_mask.to(device)

    optim = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    best_val, best_weights, no_improve = float('inf'), None, 0

    for epoch in range(1, EPOCHS+1):
        model.train()
        optim.zero_grad()
        loss = criterion(model(x_tr, ei_tr)[tr_m], y_tr[tr_m])
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optim.step()
        model.eval()
        with torch.no_grad():
            val_loss = criterion(model(x_va, ei_va)[va_m], y_va[va_m]).item()
        if val_loss < best_val - 1e-6:
            best_val, best_weights, no_improve = val_loss, deepcopy(model.state_dict()), 0
        else:
            no_improve += 1
            if PATIENCE and no_improve >= PATIENCE: break

    model.load_state_dict(best_weights)
    model.eval()
    with torch.no_grad():
        tr_pred = model(x_tr, ei_tr).cpu().numpy().flatten()
        te_pred = model(x_te, ei_te).cpu().numpy().flatten()
    tr_true = y_tr.cpu().numpy().flatten()
    te_true = y_te.cpu().numpy().flatten()
    result = {'model': label, 'n_params': count_params(model),
              **compute_metrics(tr_true, tr_pred, 'train_'),
              **compute_metrics(te_true, te_pred, 'test_')}
    if verbose:
        print(f'  {label:<45} | params={result["n_params"]:>8,} | '
              f'train RMSE={result["train_rmse"]:.4f} | '
              f'test  RMSE={result["test_rmse"]:.4f}  R2={result["test_r2"]:+.4f}')
        print_binwise(te_true, te_pred)
    return result, te_true, te_pred


def train_eval_rgcn(label, model, ei_tr, et_tr, ei_va, et_va, ei_te, et_te,
                    x_tr, y_tr, x_va, y_va, x_te, y_te,
                    tr_mask, va_mask, verbose=True):
    """Train multi-relational R-GCN with early stopping on validation MSE."""
    model = model.to(device)
    x_tr, y_tr = x_tr.to(device), y_tr.to(device)
    x_va, y_va = x_va.to(device), y_va.to(device)
    x_te, y_te = x_te.to(device), y_te.to(device)
    ei_tr, et_tr = ei_tr.to(device), et_tr.to(device)
    ei_va, et_va = ei_va.to(device), et_va.to(device)
    ei_te, et_te = ei_te.to(device), et_te.to(device)
    tr_m, va_m  = tr_mask.to(device), va_mask.to(device)

    optim = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    best_val, best_weights, no_improve = float('inf'), None, 0

    for epoch in range(1, EPOCHS+1):
        model.train()
        optim.zero_grad()
        loss = criterion(model(x_tr, ei_tr, et_tr)[tr_m], y_tr[tr_m])
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optim.step()
        model.eval()
        with torch.no_grad():
            val_loss = criterion(model(x_va, ei_va, et_va)[va_m], y_va[va_m]).item()
        if val_loss < best_val - 1e-6:
            best_val, best_weights, no_improve = val_loss, deepcopy(model.state_dict()), 0
        else:
            no_improve += 1
            if PATIENCE and no_improve >= PATIENCE: break

    model.load_state_dict(best_weights)
    model.eval()
    with torch.no_grad():
        tr_pred = model(x_tr, ei_tr, et_tr).cpu().numpy().flatten()
        te_pred = model(x_te, ei_te, et_te).cpu().numpy().flatten()
    tr_true = y_tr.cpu().numpy().flatten()
    te_true = y_te.cpu().numpy().flatten()
    result = {'model': label, 'n_params': count_params(model),
              **compute_metrics(tr_true, tr_pred, 'train_'),
              **compute_metrics(te_true, te_pred, 'test_')}
    if verbose:
        print(f'  {label:<45} | params={result["n_params"]:>8,} | '
              f'train RMSE={result["train_rmse"]:.4f} | '
              f'test  RMSE={result["test_rmse"]:.4f}  R2={result["test_r2"]:+.4f}')
        print_binwise(te_true, te_pred)
    return result, te_true, te_pred


def assemble_rgcn_graph(edge_lists):
    """Concatenate (edge_index, relation_id) pairs for RGCNConv."""
    all_ei, all_et = [], []
    for ei, rel_id in edge_lists:
        all_ei.append(ei)
        all_et.append(torch.full((ei.shape[1],), rel_id, dtype=torch.long))
    return torch.cat(all_ei, dim=1), torch.cat(all_et)


## Experiments

In [ ]:
D = x_train_tensor.shape[1]

# Shared tensors for val node features/labels (subsets of train)
x_val_tensor = x_train_tensor[val_idx]
y_val_tensor = y_train_tensor[val_idx]

# Separate val graphs built from val-only node features
X_val_proc     = X_train_proc[val_idx]
ei_sim_val     = build_similarity_edges(X_val_proc)
ei_cont_val, ei_fall_val, _ = build_global_contiguity_edges(
    train_X.index[val_idx],
    {g: i for i, g in enumerate(train_X.index[val_idx])},
    X_val_proc, SHAPE_DIR)
ei_sim_val_full = torch.unique(torch.cat([ei_sim_val, ei_fall_val], dim=1), dim=1)

results = []


### A — GCN: Similarity Only

In [ ]:
set_seed()
print('\n=== A: GCN-Similarity ===')
model_A = GCNModel(D)
res_A, te_true, te_pred_A = train_eval_gcn(
    'GCN -- similarity only', model_A,
    ei_sim_train_full, ei_sim_val_full, ei_sim_test_full,
    x_train_tensor, y_train_tensor,
    x_val_tensor,   y_val_tensor,
    x_test_tensor,  y_test_tensor,
    train_mask, val_mask)
results.append(res_A)


### B — GCN: Contiguity Only

In [ ]:
set_seed()
print('\n=== B: GCN-Contiguity ===')
model_B = GCNModel(D)
res_B, _, te_pred_B = train_eval_gcn(
    'GCN -- contiguity only', model_B,
    ei_cont_train, ei_cont_val, ei_cont_test,
    x_train_tensor, y_train_tensor,
    x_val_tensor,   y_val_tensor,
    x_test_tensor,  y_test_tensor,
    train_mask, val_mask)
results.append(res_B)


### H — GCN: Merged (sim ∪ cont, single relation)

In [ ]:
set_seed()
print('\n=== H: GCN-Merged ===')
ei_merged_train = torch.unique(torch.cat([ei_sim_train_full, ei_cont_train], dim=1), dim=1)
ei_merged_val   = torch.unique(torch.cat([ei_sim_val_full,   ei_cont_val],   dim=1), dim=1)
ei_merged_test  = torch.unique(torch.cat([ei_sim_test_full,  ei_cont_test],  dim=1), dim=1)
model_H = GCNModel(D)
res_H, _, te_pred_H = train_eval_gcn(
    'GCN -- merged (sim+cont, no relation)', model_H,
    ei_merged_train, ei_merged_val, ei_merged_test,
    x_train_tensor, y_train_tensor,
    x_val_tensor,   y_val_tensor,
    x_test_tensor,  y_test_tensor,
    train_mask, val_mask)
results.append(res_H)


### D — RGCN-SimCont (full independent weight matrices)

In [ ]:
set_seed()
print('\n=== D: RGCN-SimCont ===')
ei_D_train, et_D_train = assemble_rgcn_graph([
    (ei_sim_train_full, 0), (ei_cont_train, 1)])
ei_D_val, et_D_val = assemble_rgcn_graph([
    (ei_sim_val_full, 0), (ei_cont_val, 1)])
ei_D_test, et_D_test = assemble_rgcn_graph([
    (ei_sim_test_full, 0), (ei_cont_test, 1)])

model_D = RGCNModel(D, num_relations=2, num_bases=None)
res_D, _, te_pred_D = train_eval_rgcn(
    'R-GCN -- sim + cont [original]', model_D,
    ei_D_train, et_D_train, ei_D_val, et_D_val, ei_D_test, et_D_test,
    x_train_tensor, y_train_tensor,
    x_val_tensor,   y_val_tensor,
    x_test_tensor,  y_test_tensor,
    train_mask, val_mask)
results.append(res_D)


### D2 — RGCN-SimCont-Reg (1 shared basis matrix)

In [ ]:
set_seed()
print('\n=== D2: RGCN-SimCont-Reg (num_bases=1) ===')
model_D2 = RGCNModel(D, num_relations=2, num_bases=1)
res_D2, _, te_pred_D2 = train_eval_rgcn(
    'R-GCN -- sim + cont [2-rel, 1 bases]', model_D2,
    ei_D_train, et_D_train, ei_D_val, et_D_val, ei_D_test, et_D_test,
    x_train_tensor, y_train_tensor,
    x_val_tensor,   y_val_tensor,
    x_test_tensor,  y_test_tensor,
    train_mask, val_mask)
results.append(res_D2)


## Results Summary

In [ ]:
summary = (pd.DataFrame(results)
           [['model','n_params','train_rmse','train_r2','test_rmse','test_mae','test_r2']]
           .sort_values('test_rmse').reset_index(drop=True))
print(summary.to_string(index=False))
summary.to_csv(f'{OUT_DIR}/rgcn_results.csv', index=False)
print(f'\nSaved -> {OUT_DIR}/rgcn_results.csv')
